# 🎬 Web Scraping in Python — Oscar Winning Films Demo
### Data Science Using Python

**Recap of the journey so far:**
- BookMyShow → blocked us (403), and scraping it would go against their Terms of Use
- TMDB → needs a free API key, and that site wasn't reachable for you just now

**Today's site:** [scrapethissite.com](https://www.scrapethissite.com/pages/ajax-javascript/) — a website **built and published specifically for people to practice web scraping on**. No account, no API key, no permission needed — the site's own `robots.txt` explicitly allows scraping every page under `/pages/` (only `/lessons/` and `/faq/` are off-limits, which we don't need anyway).

The page we'll use — **"Oscar Winning Films"** — lists Academy Award-winning and nominated films from 2010–2015, with real data sourced from Wikipedia.

**Fields we'll collect:**
| Field | Meaning |
|---|---|
| Movie Name | Title of the film |
| Year | Oscar ceremony year |
| Nominations | Number of nominations received |
| Awards | Number of awards won |
| Best Picture | Whether it won Best Picture (True/False) |

> 💡 **How to use this in Colab:** Run each cell one by one using **Shift + Enter**. Don't skip ahead.

## 1. Setup

In [ ]:
# Uncomment the line below only if a library is missing (not needed on Colab)
# !pip install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd

print("All libraries imported successfully ✅")

## 2. Step 1 — Fetch the page normally... and hit a wall

Same first move as always: fetch the page and look for the data table.

In [ ]:
url = "https://www.scrapethissite.com/pages/ajax-javascript/"

response = requests.get(url)
print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")
table = soup.find("table")
rows = table.find_all("tr") if table else []
print(f"Rows found in the table: {len(rows)}")

**200 OK**, but notice the table has (almost) no rows! This page is a real-world example of **AJAX** — the film data isn't in the initial HTML at all. It only appears in your browser *after* JavaScript runs and fetches it separately, when you click a year button.

`requests` only downloads the initial HTML — it doesn't run JavaScript. So we need to find out *where* that JavaScript is getting its data from, and ask for it directly.

## 3. Step 2 — Find the real data source

If you open this page in a browser, press **F12** (DevTools) → the **Network** tab → then click a year button like **2015**, you'll see the browser quietly make a request to the *same URL*, just with two extra pieces of information added: `ajax=true` and `year=2015`. That request returns clean JSON — no HTML at all.

We can send that exact same request ourselves with `requests`, using the `params` argument.

In [ ]:
response = requests.get(url, params={"ajax": "true", "year": 2015})
print(response.status_code)
response.json()[:3]   # peek at the first 3 films

That's exactly the data we need, already as clean JSON (a Python list of dictionaries) — no BeautifulSoup required for this part! This is a really common real-world pattern: modern sites often load their content from a hidden "mini API" behind the scenes, and once you find it, scraping becomes much simpler.

## 4. Step 3 — Collect films from every year (2010–2015)

In [ ]:
YEARS = [2010, 2011, 2012, 2013, 2014, 2015]

movies = []

for year in YEARS:
    response = requests.get(url, params={"ajax": "true", "year": year})
    films = response.json()

    for f in films:
        movies.append({
            "Movie Name": f["title"].strip(),
            "Year": f["year"],
            "Nominations": f["nominations"],
            "Awards": f["awards"],
            "Best Picture": f.get("best_picture", False),
        })

print(f"Total films collected: {len(movies)}")

## 5. Step 4 — Build the DataFrame

In [ ]:
df = pd.DataFrame(movies)
df.head(10)

## 6. Step 5 — Save as CSV

In [ ]:
df.to_csv("oscar_films.csv", index=False)
print("Data saved successfully! ✅")

In [ ]:
from google.colab import files
files.download("oscar_films.csv")

## 7. Step 6 — Basic analysis

In [ ]:
# a) Number of movies collected
print("Number of movies:", len(df))

In [ ]:
# b) Number of different years scraped
print("Number of years:", df["Year"].nunique())
df["Year"].value_counts().sort_index()

In [ ]:
# c) Year with the most nominated films
print("Year with most films listed:", df["Year"].value_counts().idxmax())

In [ ]:
# d) Number of Best Picture winners
best_picture_winners = df[df["Best Picture"] == True]
print("Best Picture winners:", len(best_picture_winners))
best_picture_winners[["Movie Name", "Year"]]

In [ ]:
# e) Movies with more than 3 awards
high_award_movies = df[df["Awards"] > 3]
print("Movies with more than 3 awards:", len(high_award_movies))
high_award_movies[["Movie Name", "Year", "Awards"]]

In [ ]:
# f) Top 5 most-awarded movies
df.sort_values("Awards", ascending=False).head(5)[["Movie Name", "Year", "Nominations", "Awards"]]

## 8. 📝 Try it yourself (student exercise)

1. Add `2009` and `2008` to the `YEARS` list in Step 3 and re-run everything.
2. Filter `df` to show only movies where `Nominations` is greater than `Awards` (i.e. movies that were nominated a lot but won little).
3. Save that filtered result as `nominated.csv`.

Use the empty cell below.

In [ ]:
# 🧑‍💻 Your code here

# Step 1: Filter df where Nominations > Awards


# Step 2: Save as nominated.csv


## 9. ✅ Your complete reference script

In [ ]:
import requests
import pandas as pd

url = "https://www.scrapethissite.com/pages/ajax-javascript/"
YEARS = [2010, 2011, 2012, 2013, 2014, 2015]

movies = []
for year in YEARS:
    films = requests.get(url, params={"ajax": "true", "year": year}).json()
    for f in films:
        movies.append({
            "Movie Name": f["title"].strip(),
            "Year": f["year"],
            "Nominations": f["nominations"],
            "Awards": f["awards"],
            "Best Picture": f.get("best_picture", False),
        })

df = pd.DataFrame(movies)
df.to_csv("oscar_films.csv", index=False)

print("Number of movies:", len(df))
print("Number of years:", df["Year"].nunique())
print("Best Picture winners:", (df["Best Picture"] == True).sum())
print("Movies with more than 3 awards:", len(df[df["Awards"] > 3]))
print("\nTop 5 most-awarded movies:")
print(df.sort_values("Awards", ascending=False).head(5)[["Movie Name", "Year", "Awards"]])

## 📌 Recap

| Step | Tool | What happened |
|---|---|---|
| Fetch the page | `requests` | Website → mostly-empty HTML (AJAX site) |
| Find the hidden data request | Browser DevTools → Network tab | Discovered the real JSON source |
| Fetch the real data | `requests` + `params` | JSON response, ready to use |
| Organize it | `pandas` | List of dicts → DataFrame (table) |
| Save it | `.to_csv()` | DataFrame → CSV file |
| Ask questions | filtering, sorting, `.value_counts()` | DataFrame → answers |

### 🆚 What we've learned across all three attempts

| | BookMyShow | TMDB | scrapethissite.com |
|---|---|---|---|
| Type | Real ticketing site | Real site, official API | Practice sandbox |
| Result | 403 — blocked (against ToS) | Needs a free key, wasn't reachable today | Works instantly, no key, no blocking |
| Lesson | Not every site wants to be scraped | Prefer an official API when one exists | Practice sites exist exactly so beginners can learn safely and legally |

### ⚖️ A quick word on ethics
- This site (scrapethissite.com) was built by its creator, Hartley Brody, specifically for scraping practice — always check `robots.txt` and a site's terms before scraping *any other* real website.
- Even on a site that welcomes scraping, keep requests reasonable — don't hammer a server with rapid repeated calls.